Датасет SUES-200 из https://github.com/Reza-Zhu/SUES-200-Benchmark

In [16]:
!pip install -U --no-cache-dir gdown timm tqdm

import os
import zipfile

import gdown

file_id = '1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI'
url = f'https://drive.google.com/uc?id={file_id}'
output = '/content/sues200.zip'
extract_dir = '/content/sues200_dataset'

if not os.path.exists(extract_dir):
    gdown.download(url, output, quiet=False)
    if zipfile.is_zipfile(output):
        with zipfile.ZipFile(output, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        os.remove(output)

ValueError: mount failed

обработка снимков с дрона и спутника (изменение отдельно для спутниковых, сжатие, полярные и не полярные координаты)

In [2]:
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm


#обработка для спутниковых снимков (яркость и контраст)
def clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe_obj.apply(l)
    limg = cv2.merge((cl,a,b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def process_and_save(src_dir, dest_dir, mode='cartesian'):
    src_path = Path(src_dir)
    dest_path = Path(dest_dir)
    all_images = list(src_path.rglob('*.jpg')) + list(src_path.rglob('*.png'))
    for img_path in tqdm(all_images, desc=f"Processing {mode}"):
        img = cv2.imread(str(img_path))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if 'satellite-view' in str(img_path):
            img = clahe(img)

        if mode == 'cartesian':
            img_processed = cv2.resize(img, (224, 224))
        elif mode == 'polar':
            img_resized = cv2.resize(img, (256, 256))
            center = (128, 128)
            max_radius = 128
            img_polar = cv2.warpPolar(img_resized, (256, 256), center, max_radius, cv2.WARP_POLAR_LINEAR)
            start = (256 - 224) // 2
            img_processed = img_polar[start:start+224, start:start+224]

        rel_path = img_path.relative_to(src_path)
        save_path = dest_path / rel_path

        save_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img_processed).save(save_path)

ORIGINAL_DATASET_DIR = "/content/sues200_dataset/SUES-200-512x512"
CARTESIAN_OUT = "/content/sues_cartesian"
POLAR_OUT = "/content/sues_polar"

if os.path.exists(ORIGINAL_DATASET_DIR):
    if not os.path.exists(CARTESIAN_OUT):
        process_and_save(ORIGINAL_DATASET_DIR, CARTESIAN_OUT, mode='cartesian')
    if not os.path.exists(POLAR_OUT):
        process_and_save(ORIGINAL_DATASET_DIR, POLAR_OUT, mode='polar')

Processing polar: 100%|██████████| 40200/40200 [04:04<00:00, 164.50it/s]


добавление искажений (изменение яркости, сдвиг для полярных, повторо до 180 градусов для не полярных), разделение на обучающую и тестовую

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


#сдвиг для полярных координат
class HorizontalRoll:
    def __call__(self, img):
        shift = torch.randint(0, img.shape[2], (1,)).item()
        return torch.roll(img, shifts=shift, dims=2)

def get_transforms(mode='cartesian', is_train=True):
    if not is_train:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    base_t = [transforms.ColorJitter(brightness=0.2, contrast=0.2)]
    if mode == 'cartesian':
        base_t.insert(0, transforms.RandomRotation(180))
        base_t.append(transforms.ToTensor())
    elif mode == 'polar':
        base_t.append(transforms.ToTensor())
        base_t.append(HorizontalRoll())
    base_t.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    return transforms.Compose(base_t)

class SUESDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train', view='all'):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.images, self.labels, self.is_drone, self.heights = [], [], [], []
        loc_range = range(1, 121) if split == 'train' else range(121, 201)

        views = []
        if view in ['all', 'drone']: views.append('drone_view_512')
        if view in ['all', 'satellite']: views.append('satellite-view')

        for v_type in views:
            v_path = self.root_dir / v_type
            for img_path in v_path.rglob('*'):
                if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                    loc_num = None
                    for part in img_path.parts:
                        if part.isdigit() and len(part) == 4:
                            loc_num = int(part)
                            break
                    if loc_num is None or loc_num not in loc_range:
                        continue

                    self.images.append(str(img_path))
                    self.labels.append(loc_num)
                    drn = 'drone' in v_type
                    self.is_drone.append(drn)

                    h = 0
                    if drn:
                        for val in ['150', '200', '250', '300']:
                            if f"/{val}/" in str(img_path) or val in img_path.name:
                                h = int(val)
                                break
                    self.heights.append(h)
        print(f"Загружен {split} ({view}): {len(self.images)} фото")

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx], self.is_drone[idx], self.heights[idx], img_path

https://github.com/Gongnaiqun7/MIFT/blob/main/models/MIFT/make_model.py

первые два слоя раздельные, следующие - общие

Gem_heat для фокусировки на важном

In [10]:
import timm


class Gem_heat(nn.Module):
    def __init__(self, dim=49, p=3, eps=1e-6):
        super(Gem_heat, self).__init__()
        self.p = nn.Parameter(torch.ones(dim) * p)
        self.eps = eps

    def forward(self, x):
        x = x.permute(0, 2, 1)
        p = F.softmax(self.p, dim=0).unsqueeze(-1)
        x = torch.matmul(x, p)
        x = x.view(x.size(0), x.size(1))
        return x

class DualBranchSwin(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch_drone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0, global_pool='')
        self.branch_sat = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0, global_pool='')

        for i in range(2, len(self.branch_drone.layers)):
            self.branch_drone.layers[i] = self.branch_sat.layers[i]

        self.branch_drone.norm = self.branch_sat.norm
        self.num_features = self.branch_drone.num_features
        self.gem_pool = Gem_heat(dim=49)

    def forward(self, x, is_drone):
        feat_seq = torch.zeros(x.shape[0], 49, self.num_features, device=x.device)
        drone_mask = is_drone.bool()
        sat_mask = ~drone_mask

        if drone_mask.any():
            feat_seq[drone_mask] = self.branch_drone(x[drone_mask]).view(-1, 49, self.num_features)

        if sat_mask.any():
            feat_seq[sat_mask] = self.branch_sat(x[sat_mask]).view(-1, 49, self.num_features)

        feat = self.gem_pool(feat_seq)

        return F.normalize(feat, p=2, dim=1)

class CircleLoss(nn.Module):
    def __init__(self, m=0.25, gamma=256):
        super().__init__()
        self.m = m
        self.gamma = gamma

    def forward(self, feat, labels):
        sim = torch.matmul(feat, feat.T)
        lbl = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_m = lbl & ~torch.eye(labels.size(0), dtype=torch.bool, device=feat.device)
        neg_m = ~lbl
        sp, sn = sim[pos_m], sim[neg_m]
        if len(sp) == 0 or len(sn) == 0: return (feat * 0.0).sum()

        ap = torch.relu(-sp.detach() + 1 + self.m)
        an = torch.relu(sn.detach() + self.m)

        lp = -self.gamma * ap * (sp - (1 - self.m))
        ln = self.gamma * an * (sn - self.m)
        return torch.logsumexp(ln, dim=0) + torch.logsumexp(lp, dim=0)

оценка качества

In [5]:
def extract_features(model, loader, is_drone_flag, device):
    feats, labels, heights, paths = [], [], [], []
    with torch.no_grad():
        for img, lbl, _, h, path in tqdm(loader, desc="Извлечение признаков"):
            img = img.to(device)
            dr_flag = torch.full((img.size(0),), is_drone_flag, device=device)
            with torch.amp.autocast('cuda'):
                feat = model(img, dr_flag)
            feats.append(feat.cpu())
            labels.append(lbl)
            heights.append(h)
            paths.extend(path)
    if len(feats) == 0:
        return torch.tensor([]), torch.tensor([]), torch.tensor([]), []
    return torch.cat(feats), torch.cat(labels), torch.cat(heights), paths

def calculate_metrics(query_feat, query_labels, gallery_feat, gallery_labels):
    dist_matrix = torch.matmul(query_feat, gallery_feat.T)
    indices = torch.argsort(dist_matrix, dim=1, descending=True)

    recall_1, recall_5, recall_10 = 0, 0, 0
    all_ap = []

    for i in range(len(query_labels)):
        label = query_labels[i]
        ranked_labels = gallery_labels[indices[i]]
        correct_pos = (ranked_labels == label).nonzero(as_tuple=False).flatten()
        if len(correct_pos) > 0:
            first_match = correct_pos[0].item()
            if first_match < 1: recall_1 += 1
            if first_match < 5: recall_5 += 1
            if first_match < 10: recall_10 += 1

            ap = 0.0
            for j, pos in enumerate(correct_pos):
                ap += (j + 1) / (pos.item() + 1)
            ap /= len(correct_pos)
            all_ap.append(ap)
        else:
            all_ap.append(0.0)

    num_queries = len(query_labels)
    return {
        "R@1": recall_1 / num_queries,
        "R@5": recall_5 / num_queries,
        "R@10": recall_10 / num_queries,
        "mAP": np.mean(all_ap)
    }

def evaluate_model(model, mode, root_data):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    query_dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=False), split='test', view='drone')
    gallery_dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=False), split='test', view='satellite')

    query_loader = DataLoader(query_dataset, batch_size=32, shuffle=False, num_workers=2)
    gallery_loader = DataLoader(gallery_dataset, batch_size=32, shuffle=False, num_workers=2)

    gallery_feat, gallery_labels, _, gallery_paths = extract_features(model, gallery_loader, False, device)
    query_feat, query_labels, query_heights, query_paths = extract_features(model, query_loader, True, device)

    if query_feat.numel() == 0 or gallery_feat.numel() == 0: return None, 0.0

    results = calculate_metrics(query_feat, query_labels, gallery_feat, gallery_labels)

    print("\n")
    print(f"mAP: {results['mAP']:.4f} | R@1: {results['R@1']:.4f} | R@5: {results['R@5']:.4f} | R@10: {results['R@10']:.4f}")

    unique_heights = [150, 200, 250, 300]
    for h_val in unique_heights:
        mask = (query_heights == h_val)
        if mask.any():
            h_feat = query_feat[mask]
            h_labels = query_labels[mask]
            h_res = calculate_metrics(h_feat, h_labels, gallery_feat, gallery_labels)
            print(f"Высота {h_val}м: mAP: {h_res['mAP']:.4f} | R@1: {h_res['R@1']:.4f} | R@5: {h_res['R@5']:.4f} | R@10: {h_res['R@10']:.4f}")

    return results, results['mAP']

обучения (с сохранением лучшей модели)

In [6]:
def train_model(mode, root_data, epochs=5, batch_size=32):
    dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=True), split='train', view='all')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DualBranchSwin().to(device)

    checkpoint_path = f'/content/best_model_{mode}.pth'
    learning_rate = 1e-5
    best_map = 0.0
    criterion = CircleLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        loop = tqdm(loader, desc=f"Train Epoch {epoch+1}/{epochs} ({mode})")
        for img, labels, is_drone, _, _ in loop:
            img, labels = img.to(device), labels.to(device)
            is_drone_bool = is_drone.to(device).bool()

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                feat = model(img, is_drone_bool)
                loss = criterion(feat, labels)

            if loss.item() == 0.0: continue

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loop.set_postfix(loss=loss.item())
        scheduler.step()

        #тест после эпохи
        eval_results, current_map = evaluate_model(model, mode, root_data)

        if current_map > best_map:
            best_map = current_map
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'mAP': best_map,
            }, checkpoint_path)
            print(f"Новый лучший результат mAP для {mode.upper()}: {best_map:.4f}")

    return model

In [11]:
CARTESIAN_DIR = '/content/sues_cartesian'
if os.path.exists(CARTESIAN_DIR):
    model_cart = train_model(mode='cartesian', root_data=CARTESIAN_DIR, epochs=10)

Загружен train (all): 24120 фото


Train Epoch 1/10 (cartesian): 100%|██████████| 753/753 [04:11<00:00,  2.99it/s, loss=108]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.38it/s]




mAP: 0.7260 | R@1: 0.6186 | R@5: 0.8612 | R@10: 0.9278
Высота 150м: mAP: 0.6520 | R@1: 0.5285 | R@5: 0.7975 | R@10: 0.8792
Высота 200м: mAP: 0.7240 | R@1: 0.6148 | R@5: 0.8605 | R@10: 0.9317
Высота 250м: mAP: 0.7573 | R@1: 0.6540 | R@5: 0.8905 | R@10: 0.9510
Высота 300м: mAP: 0.7709 | R@1: 0.6770 | R@5: 0.8962 | R@10: 0.9493
Новый лучший результат mAP для CARTESIAN: 0.7260


...



mAP: 0.8350 | R@1: 0.7589 | R@5: 0.9309 | R@10: 0.9688
Высота 150м: mAP: 0.7900 | R@1: 0.7003 | R@5: 0.9035 | R@10: 0.9505
Высота 200м: mAP: 0.8314 | R@1: 0.7545 | R@5: 0.9290 | R@10: 0.9660
Высота 250м: mAP: 0.8540 | R@1: 0.7812 | R@5: 0.9437 | R@10: 0.9765
Высота 300м: mAP: 0.8647 | R@1: 0.7997 | R@5: 0.9475 | R@10: 0.9822
Новый лучший результат mAP для CARTESIAN: 0.8350


Train Epoch 10/10 (cartesian): 100%|██████████| 753/753 [03:51<00:00,  3.25it/s, loss=66.6]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:44<00:00, 11.29it/s]




mAP: 0.8359 | R@1: 0.7603 | R@5: 0.9308 | R@10: 0.9690
Высота 150м: mAP: 0.7917 | R@1: 0.7030 | R@5: 0.9038 | R@10: 0.9510
Высота 200м: mAP: 0.8314 | R@1: 0.7538 | R@5: 0.9285 | R@10: 0.9663
Высота 250м: mAP: 0.8550 | R@1: 0.7833 | R@5: 0.9445 | R@10: 0.9778
Высота 300м: mAP: 0.8656 | R@1: 0.8013 | R@5: 0.9465 | R@10: 0.9810
Новый лучший результат mAP для CARTESIAN: 0.8359


In [12]:
POLAR_DIR = '/content/sues_polar'
if os.path.exists(POLAR_DIR):
    model_polar = train_model(mode='polar', root_data=POLAR_DIR, epochs=10)

Загружен train (all): 24120 фото


Train Epoch 1/10 (polar): 100%|██████████| 753/753 [03:56<00:00,  3.19it/s, loss=102]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.47it/s]




mAP: 0.4675 | R@1: 0.3339 | R@5: 0.6175 | R@10: 0.7512
Высота 150м: mAP: 0.3777 | R@1: 0.2477 | R@5: 0.5110 | R@10: 0.6545
Высота 200м: mAP: 0.4518 | R@1: 0.3152 | R@5: 0.6050 | R@10: 0.7478
Высота 250м: mAP: 0.4948 | R@1: 0.3580 | R@5: 0.6542 | R@10: 0.7870
Высота 300м: mAP: 0.5456 | R@1: 0.4148 | R@5: 0.6997 | R@10: 0.8155
Новый лучший результат mAP для POLAR: 0.4675


...



mAP: 0.5791 | R@1: 0.4484 | R@5: 0.7375 | R@10: 0.8308
Высота 150м: mAP: 0.4926 | R@1: 0.3555 | R@5: 0.6490 | R@10: 0.7865
Высота 200м: mAP: 0.5549 | R@1: 0.4168 | R@5: 0.7300 | R@10: 0.8295
Высота 250м: mAP: 0.6204 | R@1: 0.4940 | R@5: 0.7758 | R@10: 0.8522
Высота 300м: mAP: 0.6485 | R@1: 0.5272 | R@5: 0.7953 | R@10: 0.8550


Train Epoch 10/10 (polar): 100%|██████████| 753/753 [03:51<00:00,  3.26it/s, loss=39.9]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:44<00:00, 11.14it/s]




mAP: 0.5814 | R@1: 0.4515 | R@5: 0.7398 | R@10: 0.8324
Высота 150м: mAP: 0.4950 | R@1: 0.3583 | R@5: 0.6520 | R@10: 0.7910
Высота 200м: mAP: 0.5579 | R@1: 0.4208 | R@5: 0.7318 | R@10: 0.8297
Высота 250м: mAP: 0.6209 | R@1: 0.4950 | R@5: 0.7795 | R@10: 0.8532
Высота 300м: mAP: 0.6519 | R@1: 0.5320 | R@5: 0.7960 | R@10: 0.8555


дообучение модели (с Gem_heat)

In [13]:
def fine_tune_gem(mode, root_data, epochs=5, batch_size=32):
    dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=True), split='train', view='all')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DualBranchSwin().to(device)

    old_checkpoint_path = f'/content/best_model_{mode}.pth'
    new_checkpoint_path = f'/content/best_model_{mode}_gem.pth'
    best_map = 0.0

    if os.path.exists(old_checkpoint_path):
        print(f"Загрузка весов для дообучения: {old_checkpoint_path}")
        checkpoint = torch.load(old_checkpoint_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        best_map = checkpoint.get('mAP', 0.0)

    learning_rate = 5e-6
    criterion = CircleLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        loop = tqdm(loader, desc=f"Fine-tune Gem Epoch {epoch+1}/{epochs} ({mode})")

        for img, labels, is_drone, _, _ in loop:
            img, labels = img.to(device), labels.to(device)
            is_drone_bool = is_drone.to(device).bool()

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                feat = model(img, is_drone_bool)
                loss = criterion(feat, labels)

            if loss.item() == 0.0: continue

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loop.set_postfix(loss=loss.item())
        scheduler.step()

        eval_results, current_map = evaluate_model(model, mode, root_data)
        if current_map > best_map:
            best_map = current_map
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'mAP': best_map,
            }, new_checkpoint_path)
            print(f"Новый лучший результат mAP для {mode.upper()}: {best_map:.4f}")

    return model

In [14]:
if os.path.exists(CARTESIAN_DIR):
    model_cart_gem = fine_tune_gem(mode='cartesian', root_data=CARTESIAN_DIR, epochs=5)

Загружен train (all): 24120 фото
Загрузка весов для дообучения: /content/best_model_cartesian.pth


Fine-tune Gem Epoch 1/5 (cartesian): 100%|██████████| 753/753 [03:56<00:00,  3.18it/s, loss=28.1]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.38it/s]




mAP: 0.8207 | R@1: 0.7448 | R@5: 0.9111 | R@10: 0.9586
Высота 150м: mAP: 0.7737 | R@1: 0.6835 | R@5: 0.8805 | R@10: 0.9413
Высота 200м: mAP: 0.8195 | R@1: 0.7425 | R@5: 0.9115 | R@10: 0.9540
Высота 250м: mAP: 0.8376 | R@1: 0.7638 | R@5: 0.9287 | R@10: 0.9698
Высота 300м: mAP: 0.8518 | R@1: 0.7895 | R@5: 0.9237 | R@10: 0.9692


...



mAP: 0.8201 | R@1: 0.7451 | R@5: 0.9096 | R@10: 0.9551
Высота 150м: mAP: 0.7740 | R@1: 0.6873 | R@5: 0.8768 | R@10: 0.9375
Высота 200м: mAP: 0.8195 | R@1: 0.7465 | R@5: 0.9115 | R@10: 0.9517
Высота 250м: mAP: 0.8390 | R@1: 0.7668 | R@5: 0.9267 | R@10: 0.9660
Высота 300м: mAP: 0.8478 | R@1: 0.7800 | R@5: 0.9235 | R@10: 0.9653


Fine-tune Gem Epoch 5/5 (cartesian): 100%|██████████| 753/753 [03:52<00:00,  3.23it/s, loss=50.9]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.45it/s]




mAP: 0.8257 | R@1: 0.7522 | R@5: 0.9143 | R@10: 0.9588
Высота 150м: mAP: 0.7789 | R@1: 0.6925 | R@5: 0.8840 | R@10: 0.9403
Высота 200м: mAP: 0.8247 | R@1: 0.7530 | R@5: 0.9147 | R@10: 0.9567
Высота 250м: mAP: 0.8439 | R@1: 0.7725 | R@5: 0.9310 | R@10: 0.9698
Высота 300м: mAP: 0.8552 | R@1: 0.7910 | R@5: 0.9275 | R@10: 0.9685


In [15]:
if os.path.exists(POLAR_DIR):
    model_polar = fine_tune_gem(mode='polar', root_data=POLAR_DIR, epochs=5)

Загружен train (all): 24120 фото
Загрузка весов для дообучения: /content/best_model_polar.pth


Fine-tune Gem Epoch 1/5 (polar): 100%|██████████| 753/753 [03:45<00:00,  3.34it/s, loss=99]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.44it/s]




mAP: 0.5787 | R@1: 0.4507 | R@5: 0.7313 | R@10: 0.8241
Высота 150м: mAP: 0.4877 | R@1: 0.3483 | R@5: 0.6490 | R@10: 0.7810
Высота 200м: mAP: 0.5525 | R@1: 0.4155 | R@5: 0.7235 | R@10: 0.8187
Высота 250м: mAP: 0.6229 | R@1: 0.5028 | R@5: 0.7675 | R@10: 0.8455
Высота 300м: mAP: 0.6516 | R@1: 0.5363 | R@5: 0.7853 | R@10: 0.8510


...



mAP: 0.5764 | R@1: 0.4506 | R@5: 0.7241 | R@10: 0.8212
Высота 150м: mAP: 0.4982 | R@1: 0.3670 | R@5: 0.6430 | R@10: 0.7833
Высота 200м: mAP: 0.5489 | R@1: 0.4145 | R@5: 0.7140 | R@10: 0.8157
Высота 250м: mAP: 0.6149 | R@1: 0.4940 | R@5: 0.7592 | R@10: 0.8415
Высота 300м: mAP: 0.6437 | R@1: 0.5268 | R@5: 0.7800 | R@10: 0.8442


Fine-tune Gem Epoch 5/5 (polar): 100%|██████████| 753/753 [03:46<00:00,  3.33it/s, loss=42.8]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:43<00:00, 11.52it/s]




mAP: 0.5821 | R@1: 0.4574 | R@5: 0.7285 | R@10: 0.8236
Высота 150м: mAP: 0.5024 | R@1: 0.3705 | R@5: 0.6452 | R@10: 0.7880
Высота 200м: mAP: 0.5551 | R@1: 0.4225 | R@5: 0.7208 | R@10: 0.8163
Высота 250м: mAP: 0.6208 | R@1: 0.5008 | R@5: 0.7628 | R@10: 0.8435
Высота 300м: mAP: 0.6500 | R@1: 0.5357 | R@5: 0.7853 | R@10: 0.8465
